# [WIP] Proposed Redesign of SB-MOABB DataIO


In [1]:
%load_ext line_profiler
%load_ext memory_profiler

In [2]:
%%capture
!pip install speechbrain moabb mne mne_bids braindecode

# Extend DynamicItemDataset to support BIDS and MOABB datasets


In [3]:
import json
from pathlib import Path
from typing import Any

import mne
import moabb
import numpy as np
from mne_bids import BIDSPath, get_bids_path_from_fname, read_raw_bids
from moabb.datasets import BNCI2014_001
from moabb.datasets import download as dl
from moabb.datasets.base import BaseDataset as BaseMOABBDataset
from moabb.datasets.bids_interface import camel_to_kebab_case
from typing_extensions import Optional, Self

mne.set_log_level(verbose=False)
moabb.set_log_level(level="ERROR")

In [4]:
import speechbrain as sb
from speechbrain.dataio.dataset import DynamicItemDataset

/home/radicalshadow/Projects/Research/benchmarks/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class RawEEGDataset(DynamicItemDataset):

    def __init__(
        self,
        data,
        dynamic_items=[],
        output_keys=[],
        preload=False,
        warmup_raw_cache=True,
    ):
        # TODO set dynamic_item to load data
        self._cached_raws = {}

        dynamic_items = [
            self._make_load_raw_dynamic_item(preload)
        ] + dynamic_items
        super().__init__(
            data, dynamic_items=dynamic_items, output_keys=output_keys
        )

        if warmup_raw_cache:
            with self.output_keys_as(["raw"]):
                for _ in self:
                    pass

    def _make_load_raw_dynamic_item(self, preload):
        @sb.utils.data_pipeline.takes("fpath")
        @sb.utils.data_pipeline.provides("info", "raw")
        def _load_raw(fpath):  # -> Generator[Any, Any, None]:
            if fpath in self._cached_raws:
                raw = self._cached_raws[fpath]
            else:
                bids_path = get_bids_path_from_fname(fpath)
                raw = read_raw_bids(
                    bids_path, extra_params=dict(preload=preload), verbose=0
                )
                self._cached_raws[fpath] = raw

            yield raw.info
            yield raw

        return _load_raw

    @classmethod
    def from_bids(
        cls,
        bids_path: Path | str | BIDSPath,
        json_path: str | Path,
        subjects=None,
        **cls_kwargs,
    ) -> Self:
        """Creates a DynamicItemDataset from a BIDS EEG Dataset."""
        if not isinstance(bids_path, BIDSPath):
            bids_path = BIDSPath(root=bids_path)
        json_data = cls.load_or_create_json_data_from_bids(
            bids_path, json_path, subjects=subjects
        )

        return cls(data=json_data, **cls_kwargs)

    @classmethod
    def from_moabb(
        cls,
        dataset: BaseMOABBDataset,
        json_path: str | Path,
        subjects=None,
        save_path: Optional[str] = None,
        **cls_kwargs,
    ) -> Self:
        """Creates a DynamicItemDataset from a MOABB Dataset, by first
        converting it to BIDS format."""
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)

            return cls(data=json_data, **cls_kwargs)

        mne_path = Path(dl.get_dataset_path(dataset.code, save_path))

        cache_dir = f"MNE-BIDS-{camel_to_kebab_case(dataset.code)}"
        cache_path = mne_path / cache_dir

        subject_list = (
            subjects if subjects is not None else dataset.subject_list
        )
        dataset.download(subject_list)

        for sub in subject_list:
            dataset.get_data(
                subjects=[sub],
                cache_config=dict(use=True, save_raw=True, path=mne_path),
            )

        return cls.from_bids(
            bids_path=cache_path,
            json_path=json_path,
            subjects=subjects,
            **cls_kwargs,
        )

    @classmethod
    def load_or_create_json_data_from_bids(
        cls,
        bids_path: BIDSPath,
        json_path: str | Path,
        subjects=None,
    ) -> dict[str, Any]:
        json_path = Path(json_path)
        if json_path.exists():
            with json_path.open() as fp:
                json_data = json.load(fp)
        else:
            json_data = cls.json_data_from_bids_path(bids_path)

            with json_path.open("w") as fp:
                json.dump(json_data, fp)

        if subjects is not None:
            json_data = {
                uid: data
                for uid, data in json_data.items()
                if data["subject"] in subjects
            }

        return json_data

    @classmethod
    def json_data_from_bids_path(cls, bids_path) -> dict[str, Any]:
        json_data = {}

        for path in bids_path.update(suffix="eeg").match(ignore_json=True):
            uid = path.fpath.name
            json_data[uid] = path.entities
            json_data[uid]["fpath"] = str(path.fpath)
        return json_data


class EpochedEEGDataset(RawEEGDataset):
    """Breaks the raw EEG signals up into epochs."""

    def __init__(
        self,
        data,
        tmin=0,
        tmax=None,
        dynamic_items=[],
        output_keys=[],
        **kwargs,
    ):
        # TODO set dynamic_item to load data
        dynamic_items = [
            self._make_load_epoch_dynamic_item(tmin, tmax)
        ] + dynamic_items
        super().__init__(
            data, dynamic_items=dynamic_items, output_keys=output_keys, **kwargs
        )

    def _make_load_epoch_dynamic_item(self, tmin, tmax):

        @sb.utils.data_pipeline.takes("raw", "onset")
        @sb.utils.data_pipeline.provides("epoch")
        def _load_epoch(raw: mne.io.RawArray, onset):
            onset_time = onset / raw.info["sfreq"]

            tmin_index = int((onset_time + tmin) * raw.info["sfreq"])
            tmax_index = (
                int((onset_time + tmax) * raw.info["sfreq"])
                if tmax is not None
                else -1
            )

            return raw._getitem(
                (slice(None), slice(tmin_index, tmax_index)),
                return_times=False,
            ).astype(np.float32)

        return _load_epoch

    @classmethod
    def json_data_from_bids_path(cls, bids_path) -> dict[str, Any]:
        raw_json_data = super().json_data_from_bids_path(bids_path)

        json_data = {}
        for uid, sample in raw_json_data.items():
            bids_path = get_bids_path_from_fname(sample["fpath"])
            raw = read_raw_bids(
                bids_path, extra_params=dict(preload=False), verbose=False
            )
            stim_channels = mne.utils._get_stim_channel(
                None, raw.info, raise_error=False
            )
            if len(stim_channels) > 0:
                # returns empty array if none found
                events = mne.find_events(raw, shortest_event=0, verbose=False)
                event_id = {}
            else:
                events, event_id = mne.events_from_annotations(
                    raw, verbose=False
                )

            event_id = {v: k for k, v in event_id.items()}

            for onset, _, event in events:
                label = event_id.get(event, int(event))
                event_sample = dict(**sample, label=label, onset=int(onset))
                event_uid = f"{uid}/{label}/{onset}"
                json_data[event_uid] = event_sample

        return json_data

    @classmethod
    def from_moabb(
        cls,
        dataset: BaseMOABBDataset,
        json_path: str | Path,
        subjects=None,
        save_path: str | None = None,
        **cls_kwargs,
    ) -> Self:
        if "tmin" not in cls_kwargs:
            cls_kwargs.update(tmin=0)
        if "tmax" not in cls_kwargs:
            cls_kwargs.update(tmax=dataset.interval[1] - dataset.interval[0])

        return super().from_moabb(
            dataset, json_path, subjects, save_path, **cls_kwargs
        )

In [6]:
%%memit
dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    tmin=0,
    tmax=4.0,
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
    ],
)

for _ in dataset:
    pass


peak memory: 676.01 MiB, increment: 42.46 MiB


In [7]:
%%time
for _ in dataset:
    pass

CPU times: user 1.69 s, sys: 102 ms, total: 1.79 s
Wall time: 1.79 s


In [8]:
%%memit
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import create_windows_from_events

raw_braindecode_dataset = MOABBDataset("BNCI2014_001", subject_ids=None)
epoched_braindecode_dataset = create_windows_from_events(raw_braindecode_dataset)

for sample in epoched_braindecode_dataset:
    pass

peak memory: 2958.96 MiB, increment: 2282.95 MiB


In [9]:
%%timeit

for sample in epoched_braindecode_dataset:
    pass

163 ms ± 755 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%memit
dataset = EpochedEEGDataset.from_moabb(
    BNCI2014_001(),
    "data/MNE-BIDS-bnci2014-001-epoched.json",
    save_path="data",
    tmin=0,
    tmax=4.0,
    output_keys=[
        "label",
        "subject",
        "session",
        "epoch",
    ],
    preload=True
)

for _ in dataset:
    pass


peak memory: 4620.03 MiB, increment: 1779.12 MiB


In [11]:
%%timeit
for _ in dataset:
    pass

179 ms ± 1.03 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
print(dataset[0]["epoch"].shape, epoched_braindecode_dataset[0][0].shape)
np.allclose(dataset[0]["epoch"], epoched_braindecode_dataset[0][0][:22])

(22, 1000) (26, 1000)


True